## DamageDiffusion - Colab Training

---

### Configuration

In [ ]:
GITHUB_REPO = "https://github.com/youlkar/damage-diffusion.git"
PROJECT_DIR = "/content/drive/MyDrive/DamageDiffusion"
NUM_EPOCHS = 100

### Mount Google Drive

In [ ]:
from google.colab import drive
import os
import time

# Mount with timeout handling
drive.mount('/content/drive')
os.makedirs(PROJECT_DIR, exist_ok=True)

print("Google Drive mounted")
print(f"Project directory: {PROJECT_DIR}")

# Verify connection with retry
max_retries = 3
for i in range(max_retries):
    try:
        # Test Drive connection
        test_files = os.listdir('/content/drive/MyDrive')
        print(f"Drive connection verified ({len(test_files)} items in MyDrive)")
        break
    except OSError as e:
        if i < max_retries - 1:
            print(f"Drive connection issue, retrying... ({i+1}/{max_retries})")
            time.sleep(2)
        else:
            print("WARNING: Drive connection unstable. You may need to restart runtime.")

### Verify GPU

In [ ]:
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {gpu_name}")
    print(f"VRAM: {gpu_memory:.1f} GB")
    
    # Auto-detect optimal batch size
    if 'T4' in gpu_name:
        BATCH_SIZE = 12
        print("Recommended batch size: 12 (T4)")
    elif 'V100' in gpu_name:
        BATCH_SIZE = 24
        print("Recommended batch size: 24 (V100)")
    elif 'A100' in gpu_name or 'A10' in gpu_name:
        BATCH_SIZE = 48
        print("Recommended batch size: 48 (A100)")
    else:
        BATCH_SIZE = 16
        print("Recommended batch size: 16")
else:
    print("No GPU detected!")
    BATCH_SIZE = 8

### Clone Code from GitHub

First time: Clones repository  
Updates: Pulls latest changes

In [ ]:
import os

# Check if already cloned
if os.path.exists(f"{PROJECT_DIR}/.git"):
    print("Repository already exists. Pulling latest changes...")
    %cd {PROJECT_DIR}
    !git pull
    print("Code updated from GitHub")
else:
    print("Cloning repository from GitHub...")
    !git clone {GITHUB_REPO} {PROJECT_DIR}
    print("Code cloned from GitHub")

# Verify backend folder exists
if os.path.exists(f"{PROJECT_DIR}/backend"):
    print(f"Backend code found at {PROJECT_DIR}/backend")
else:
    print("Backend folder not found!")
    print("Make sure you pushed the 'backend/' folder to GitHub")

### Download Dataset from HuggingFace

Downloads from: https://huggingface.co/datasets/varcoder/crack-segmentation-dataset

In [ ]:
import os

dataset_dir = f"{PROJECT_DIR}/data/crack_segmentation_dataset"

# Check if dataset exists
if os.path.exists(dataset_dir) and os.path.exists(f"{dataset_dir}/train/images"):
    print("Dataset already exists in Google Drive!")
    print(f"  Location: {dataset_dir}")
    
    # Count files
    train_images = len(os.listdir(f"{dataset_dir}/train/images")) if os.path.exists(f"{dataset_dir}/train/images") else 0
    train_masks = len(os.listdir(f"{dataset_dir}/train/masks")) if os.path.exists(f"{dataset_dir}/train/masks") else 0
    test_images = len(os.listdir(f"{dataset_dir}/test/images")) if os.path.exists(f"{dataset_dir}/test/images") else 0
    test_masks = len(os.listdir(f"{dataset_dir}/test/masks")) if os.path.exists(f"{dataset_dir}/test/masks") else 0
    
    print(f"  Train images: {train_images} | Train masks: {train_masks}")
    print(f"  Test images: {test_images} | Test masks: {test_masks}")
    
    if train_images != train_masks:
        print(f"\nWARNING: Mismatch! {train_images - train_masks} images missing masks")
else:
    print("Downloading dataset from HuggingFace (2-3 min)...\n")
    
    # Install datasets library
    !pip install -q datasets pillow
    
    from datasets import load_dataset
    
    print("[1/3] Loading dataset metadata...")
    dataset = load_dataset("varcoder/crack-segmentation-dataset")
    
    print(f"[2/3] Saving images to Google Drive...")
    print(f"  Train samples: {len(dataset['train'])}")
    print(f"  Test samples: {len(dataset['test'])}")
    
    # Create directories
    os.makedirs(f"{dataset_dir}/train/images", exist_ok=True)
    os.makedirs(f"{dataset_dir}/train/masks", exist_ok=True)
    os.makedirs(f"{dataset_dir}/test/images", exist_ok=True)
    os.makedirs(f"{dataset_dir}/test/masks", exist_ok=True)
    
    # Process train split
    print("\n  Processing train split...")
    for i, item in enumerate(dataset['train']):
        # HuggingFace uses 'pixel_values' for image and 'label' for mask
        img = item['pixel_values']  # PIL Image
        mask = item['label']  # PIL Image
        
        img.save(f"{dataset_dir}/train/images/{i:05d}.jpg")
        mask.save(f"{dataset_dir}/train/masks/{i:05d}.jpg")
        
        if (i+1) % 1000 == 0:
            print(f"    Processed {i+1} images...")
    
    # Process test split
    print("\n  Processing test split...")
    for i, item in enumerate(dataset['test']):
        img = item['pixel_values']
        mask = item['label']
        
        img.save(f"{dataset_dir}/test/images/{i:05d}.jpg")
        mask.save(f"{dataset_dir}/test/masks/{i:05d}.jpg")
    
    train_images = len(os.listdir(f"{dataset_dir}/train/images"))
    train_masks = len(os.listdir(f"{dataset_dir}/train/masks"))
    test_images = len(os.listdir(f"{dataset_dir}/test/images"))
    test_masks = len(os.listdir(f"{dataset_dir}/test/masks"))
    
    print(f"\n[3/3] Dataset saved to Google Drive!")
    print(f"  Location: {dataset_dir}")
    print(f"  Train: {train_images} images, {train_masks} masks")
    print(f"  Test: {test_images} images, {test_masks} masks")
    print("  Dataset will persist across all Colab sessions.")

### Install Dependencies

In [ ]:
print("Installing dependencies...")

# Method 1: Try from local copy (fastest, avoids Drive connection issues)
import os
try:
    !cp {PROJECT_DIR}/backend/requirements.txt /content/requirements.txt
    !pip install -q -r /content/requirements.txt
    !rm /content/requirements.txt
    print("Dependencies installed from local copy")
except Exception as e:
    print(f"Local copy failed, installing directly from GitHub...")
    # Method 2: Fallback to direct GitHub installation
    !pip install -q torch>=2.0.0 torchvision>=0.15.0 numpy>=1.24.0 pillow>=9.5.0
    !pip install -q diffusers>=0.25.0 accelerate>=0.25.0 transformers>=4.35.0
    !pip install -q tensorboard>=2.15.0 tqdm>=4.66.0
    !pip install -q pytorch-fid>=0.3.0 scikit-learn>=1.3.0 scipy>=1.11.0
    !pip install -q pandas>=2.0.0 matplotlib>=3.7.0 seaborn>=0.12.0
    print("Dependencies installed from direct specification")

### Test Setup

In [ ]:
%cd {PROJECT_DIR}/backend

# Run test script
!python test_setup.py

### Start Training

In [ ]:
%cd {PROJECT_DIR}/backend

# Start fresh training
!python train.py --config colab --batch_size {BATCH_SIZE} --num_epochs {NUM_EPOCHS} --data_root {PROJECT_DIR}/data/crack_segmentation_dataset --checkpoint_dir {PROJECT_DIR}/checkpoints --log_dir {PROJECT_DIR}/logs

### Resume Training After Disconnect

In [ ]:
import glob
import os

%cd {PROJECT_DIR}/backend

# Find latest checkpoint
checkpoint_pattern = f"{PROJECT_DIR}/checkpoints/checkpoint_epoch_*.pt"
checkpoints = sorted(glob.glob(checkpoint_pattern))

if checkpoints:
    latest_checkpoint = checkpoints[-1]
    epoch_num = os.path.basename(latest_checkpoint).split('_')[-1].replace('.pt', '')
    print(f"Found checkpoint: {os.path.basename(latest_checkpoint)}")
    print(f"Resuming from epoch {epoch_num}")
    print()
    
    # Resume training
    !python train.py --resume {latest_checkpoint} --config colab --batch_size {BATCH_SIZE} --num_epochs {NUM_EPOCHS} --data_root {PROJECT_DIR}/data/crack_segmentation_dataset --checkpoint_dir {PROJECT_DIR}/checkpoints --log_dir {PROJECT_DIR}/logs
else:
    print("No checkpoints found.")
    print("Use the 'Start Training' cell to start training from scratch.")

### Monitor Training - View Samples

In [ ]:
from IPython.display import Image, display
import glob

# Find latest samples
sample_pattern = f"{PROJECT_DIR}/samples/samples_epoch_*.png"
samples = sorted(glob.glob(sample_pattern))

if samples:
    latest = samples[-1]
    epoch = os.path.basename(latest).split('_')[-1].replace('.png', '')
    print(f"Latest samples (Epoch {epoch}):")
    display(Image(latest, width=800))
else:
    print("No samples generated yet.")
    print("Samples are generated every 5 epochs.")
    print("Wait for epoch 5, then re-run this cell.")

### Monitor Training - TensorBoard

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {PROJECT_DIR}/logs

### Check Training Progress

In [ ]:
import glob
import os

# Check checkpoints
checkpoints = sorted(glob.glob(f"{PROJECT_DIR}/checkpoints/checkpoint_epoch_*.pt"))
print(f"Training Progress\n" + "="*50)

if checkpoints:
    latest = checkpoints[-1]
    epoch = os.path.basename(latest).split('_')[-1].replace('.pt', '')
    print(f"Current epoch: {epoch}/{NUM_EPOCHS}")
    print(f"Progress: {int(epoch)/NUM_EPOCHS*100:.1f}%")
    print(f"Checkpoints saved: {len(checkpoints)}")
    print(f"\nLatest checkpoint: {os.path.basename(latest)}")
else:
    print("No checkpoints found yet.")
    print("Training may have just started.")

# Check if best model exists
if os.path.exists(f"{PROJECT_DIR}/checkpoints/best_model.pt"):
    print("\nBest model saved")

# Check samples
samples = glob.glob(f"{PROJECT_DIR}/samples/*.png")
if samples:
    print(f"\nSamples generated: {len(samples)}")

### Generate Images After Training - Inference

In [ ]:
%cd {PROJECT_DIR}/backend

import glob
import os

# Find the best available checkpoint
checkpoint_dir = f"{PROJECT_DIR}/checkpoints"
best_checkpoint = f"{checkpoint_dir}/best_model.pt"
final_checkpoint = f"{checkpoint_dir}/final_model.pt"

# Priority: best_model.pt > final_model.pt > latest epoch checkpoint
if os.path.exists(best_checkpoint):
    CHECKPOINT = best_checkpoint
    print(f"Using best model: {CHECKPOINT}")
elif os.path.exists(final_checkpoint):
    CHECKPOINT = final_checkpoint
    print(f"Using final model: {CHECKPOINT}")
else:
    # Find latest epoch checkpoint
    epoch_checkpoints = sorted(glob.glob(f"{checkpoint_dir}/checkpoint_epoch_*.pt"))
    if epoch_checkpoints:
        CHECKPOINT = epoch_checkpoints[-1]
        epoch_num = os.path.basename(CHECKPOINT).split('_')[-1].replace('.pt', '')
        print(f"Using latest checkpoint from epoch {epoch_num}: {CHECKPOINT}")
    else:
        print("ERROR: No checkpoints found!")
        print("Please run training first")
        CHECKPOINT = None

if CHECKPOINT:
    TEST_MASK = f"{PROJECT_DIR}/data/crack_segmentation_dataset/test/masks/00001.jpg"
    OUTPUT_DIR = f"{PROJECT_DIR}/generated"

    !python inference.py --checkpoint {CHECKPOINT} --mask_path {TEST_MASK} --output_dir {OUTPUT_DIR} --num_samples 4 --num_steps 100

    # Display result
    from IPython.display import Image, display
    import glob

    generated = glob.glob(f"{OUTPUT_DIR}/*.png")
    if generated:
        print("\nGenerated images:")
        display(Image(generated[-1], width=800))

### Download Results

In [ ]:
from google.colab import files
import shutil
import os

# What to download
print("Creating download archives...\n")

# 1. Best model
if os.path.exists(f"{PROJECT_DIR}/checkpoints/best_model.pt"):
    print("Downloading best model...")
    files.download(f"{PROJECT_DIR}/checkpoints/best_model.pt")
else:
    print("Best model not found")

# 2. All checkpoints (zipped)
if os.path.exists(f"{PROJECT_DIR}/checkpoints"):
    print("Creating checkpoints archive...")
    shutil.make_archive('/content/checkpoints', 'zip', f"{PROJECT_DIR}/checkpoints")
    files.download('/content/checkpoints.zip')

# 3. Generated samples (zipped)
if os.path.exists(f"{PROJECT_DIR}/samples"):
    print("Creating samples archive...")
    shutil.make_archive('/content/samples', 'zip', f"{PROJECT_DIR}/samples")
    files.download('/content/samples.zip')

print("\nDownloads complete!")

### Utilities - Update Code from GitHub

In [ ]:
%cd {PROJECT_DIR}
!git pull
print("Code updated from GitHub")

### Clear Old Checkpoints (Save Drive Space)

In [ ]:
# Keep only best_model.pt and final_model.pt
# Delete intermediate checkpoints

import glob
import os

intermediate = glob.glob(f"{PROJECT_DIR}/checkpoints/checkpoint_epoch_*.pt")
print(f"Found {len(intermediate)} intermediate checkpoints")
print("\nDelete them? (keeps best_model.pt and final_model.pt)")
print("Uncomment the line below to delete:")

# Uncomment to delete:
# for cp in intermediate:
#     os.remove(cp)
#     print(f"Deleted: {os.path.basename(cp)}")

### Check GPU Memory Usage

In [ ]:
!nvidia-smi